# Local one-day pure-space Vecchia comparison

Created 2026-05-24.

This notebook runs the default pure-space isotropic Matérn cluster Vecchia B2 implementation on one generated July simulation day, i.e. 8 hourly spatial fields treated as independent pure-space replicates sharing the same covariance parameters.

- Default 4x4 cluster max-min Vecchia: `kernels_space_iso_cluster_052426.py`, neighbor block count `B=2`

The simulation truth is anisotropic, while both tested models are isotropic. The reference isotropic range used here is `sqrt(range_lat * range_lon)`.

In [ ]:
from pathlib import Path
import sys, os, gc, json, time
import importlib.util
from types import SimpleNamespace

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

ROOT = Path('/Users/joonwonlee/Documents/GEMS_TCO-1')
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

HELPER_PATH = ROOT / 'Exercises/st_model/day/amarel_simulation/pure_space/fit_sim_july_pure_space_vecchia_compare_052426.py'
spec = importlib.util.spec_from_file_location('pure_space_compare_helper_052426', HELPER_PATH)
helper = importlib.util.module_from_spec(spec)
spec.loader.exec_module(helper)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
helper.DEVICE = DEVICE
helper.DTYPE = torch.float64
print('device:', DEVICE)
print('helper:', HELPER_PATH)

## Configuration

`DATA_ROOT` should point to the output directory from:

```bash
python simulate_data/generate_july_st_circulant_real_locations_2022_2025.py --output-dir <DATA_ROOT>
```

It should contain subfolders like `2022_july_st_circulant/sim_july2022_st_circulant_real_locations.pkl`.

In [ ]:
YEAR = '2022'
DAY_IDXS = '0'        # one 8-hour day block; use '0,1,2' or 'all' for more replicates
NUM_DAY_ASSETS = 1    # keep 1 for the local one-day test
N_RESTARTS = 1        # increase to 3+ if you want nonzero p90-p10 over restarts

OUT_DIR = ROOT / 'Exercises/st_model/day/local_computer/pure_space/log/sim_july_one_day_vecchia_compare_052426'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Use already generated simulation output from
# simulate_data/generate_july_st_circulant_real_locations_2022_2025.py.
# Local generated data found under /Users/joonwonlee/Documents/GEMS_DATA/simulation.
SIM_SMOOTH_TAG = 'smooth0p5'  # available locally: smooth0p3, smooth0p5, smooth1p0
MANUAL_DATA_ROOT = None   # e.g. Path('/path/to/july_st_circulant_realpattern')

DATA_ROOT_CANDIDATES = [
    Path('/Users/joonwonlee/Documents/GEMS_DATA/simulation') / f'july_st_circulant_realpattern_{SIM_SMOOTH_TAG}',
    ROOT / 'outputs/day/sim_data/july_st_circulant_realpattern',
    ROOT / 'outputs/sim_data/july_st_circulant_realpattern',
    ROOT / 'exercise_output/sim_data/july_st_circulant_realpattern',
    ROOT / 'simulate_data/july_st_circulant_realpattern',
    Path('/home/jl2815/tco/exercise_output/sim_data/july_st_circulant_realpattern'),
]

def has_generated_year(root, year):
    root = Path(root)
    return (root / f'{year}_july_st_circulant' / f'sim_july{year}_st_circulant_real_locations.pkl').exists()

DATA_ROOT = Path(MANUAL_DATA_ROOT) if MANUAL_DATA_ROOT is not None else next((p for p in DATA_ROOT_CANDIDATES if has_generated_year(p, YEAR)), None)

if DATA_ROOT is None:
    checked = '\n'.join(str(p) for p in DATA_ROOT_CANDIDATES)
    raise FileNotFoundError('Could not find pre-generated simulation data. Set MANUAL_DATA_ROOT manually. Checked:\n' + checked)

CLUSTER_BLOCKS = [2, 3, 4, 5, 6, 7]

# Local sanity-check controls.  These match the Amarel script but can be lowered if CPU is slow.
LBFGS_STEPS = 5
LBFGS_EVAL = 15
POINT_TARGET_CHUNK_SIZE = 4096
CLUSTER_TARGET_CHUNK_SIZE = 96

print('DATA_ROOT:', DATA_ROOT)
print('OUT_DIR:', OUT_DIR)

In [ ]:
args = SimpleNamespace(
    num_sims=NUM_DAY_ASSETS,
    seed=123,
    years=YEAR,
    day_idxs=DAY_IDXS,
    max_asset_days=NUM_DAY_ASSETS,
    asset_sampling='cycle',
    sim_data_root=DATA_ROOT,
    data_kind='real_locations',
    lat_range=[-3.0, 2.0],
    lon_range=[121.0, 131.0],
    cluster_neighbor_blocks=CLUSTER_BLOCKS,
    cluster_block_shape=(4, 4),
    mean_design='latlon',
    smooth_fallback=0.5,
    cluster_target_chunk_size=CLUSTER_TARGET_CHUNK_SIZE,
    min_target_points=1,
    lbfgs_steps=LBFGS_STEPS,
    lbfgs_eval=LBFGS_EVAL,
    lbfgs_hist=10,
    grad_tol=1e-5,
    init_noise=0.25,
    summary_every=1,
    center_response=True,
    require_cuda=False,
    resume=False,
    out_dir=OUT_DIR,
)

helper.set_seed(args.seed)
asset_bank, truth = helper.build_asset_bank(args)
specs = helper.make_specs(args)

print('n_assets:', len(asset_bank))
print('n_specs:', len(specs))
print('truth isotropic reference:', {k: truth[k] for k in helper.PARAMS})
pd.DataFrame(specs)

## Fit one generated day

This runs all requested conditioning settings on the selected day. With `N_RESTARTS=1`, p90-p10 within each setting is zero because there is only one fit per setting. Increase `N_RESTARTS` or `DAY_IDXS` if you want distribution summaries.

In [ ]:
rows = []
fit_id = 0
for asset_i, asset in enumerate(asset_bank[:NUM_DAY_ASSETS], start=1):
    source_map_raw = {
        k: v.to(device=DEVICE, dtype=torch.float64).contiguous()
        for k, v in asset['source_map'].items()
    }
    grid_coords_np = np.asarray(asset['grid_coords_np'], dtype=np.float64)
    n_valid = sum(int(torch.isfinite(v[:, 2]).sum().item()) for v in source_map_raw.values())
    print(f"asset {asset_i}: year={asset['year']} day_idx={asset['day_idx']} n_grid={asset['n_grid']:,} n_valid={n_valid:,}")

    for restart in range(N_RESTARTS):
        rng = np.random.default_rng(args.seed + 10_000 * asset_i + restart)
        for model_spec in specs:
            fit_id += 1
            t0 = time.time()
            print(f"\n[{fit_id}/{len(asset_bank[:NUM_DAY_ASSETS]) * N_RESTARTS * len(specs)}] {model_spec['spec_name']} restart={restart}")
            try:
                row = helper.fit_model(model_spec, source_map_raw, grid_coords_np, truth, args, rng)
                row.update({
                    'fit_id': fit_id,
                    'asset_i': asset_i,
                    'restart': restart,
                    'asset_year': asset['year'],
                    'asset_day_idx': asset['day_idx'],
                    'n_valid': n_valid,
                    'n_grid': asset['n_grid'],
                    'valid_by_t': json.dumps(asset['valid_by_t']),
                })
                print('done', {k: round(row[k], 5) for k in ['overall_rmsre', 'median_abs_error', 'total_s']})
            except Exception as exc:
                row = {
                    'fit_id': fit_id,
                    'asset_i': asset_i,
                    'restart': restart,
                    'model_family': model_spec['model_family'],
                    'spec_name': model_spec['spec_name'],
                    'asset_year': asset['year'],
                    'asset_day_idx': asset['day_idx'],
                    'n_valid': n_valid,
                    'n_grid': asset['n_grid'],
                    'error': repr(exc),
                }
                print('ERROR:', repr(exc))
            rows.append(row)
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

df = pd.DataFrame(rows)
raw_csv = OUT_DIR / 'local_one_day_all_fits.csv'
df.to_csv(raw_csv, index=False)
print('saved:', raw_csv)
df[['model_family', 'spec_name', 'overall_rmsre', 'median_abs_error', 'total_s', 'error'] if 'error' in df else ['model_family', 'spec_name', 'overall_rmsre', 'median_abs_error', 'total_s']]

In [ ]:
model_summary = helper.make_model_summary(df.assign(error=df.get('error', '')))
param_summary = helper.make_param_summary(df.assign(error=df.get('error', '')), truth)

model_csv = OUT_DIR / 'local_one_day_model_summary.csv'
param_csv = OUT_DIR / 'local_one_day_param_summary.csv'
model_summary.to_csv(model_csv, index=False)
param_summary.to_csv(param_csv, index=False)

print('saved:', model_csv)
print('saved:', param_csv)

display(model_summary[[
    'model_family', 'spec_name', 'n',
    'overall_rmsre_median', 'overall_rmsre_p90_p10',
    'median_abs_error_median', 'total_s_median', 'n_cond_points_nominal_median'
]].sort_values('overall_rmsre_median'))

display(param_summary[[
    'model_family', 'spec_name', 'parameter', 'n', 'rmsre',
    'median_abs_error', 'p90_p10_abs_error', 'median_re', 'estimate_median'
]].sort_values(['parameter', 'rmsre']).head(50))

In [ ]:
plot_df = model_summary.sort_values('overall_rmsre_median').copy()
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].barh(plot_df['spec_name'], plot_df['overall_rmsre_median'])
axes[0].invert_yaxis()
axes[0].set_xlabel('overall RMSRE median')
axes[0].set_title('Accuracy')

axes[1].barh(plot_df['spec_name'], plot_df['total_s_median'])
axes[1].invert_yaxis()
axes[1].set_xlabel('seconds median')
axes[1].set_title('Runtime')

fig.tight_layout()
png = OUT_DIR / 'local_one_day_model_comparison.png'
fig.savefig(png, dpi=160, bbox_inches='tight')
print('saved:', png)
plt.show()

In [ ]:
best = model_summary.sort_values('overall_rmsre_median').head(5)
best[['model_family', 'spec_name', 'n', 'overall_rmsre_median', 'median_abs_error_median', 'total_s_median', 'n_cond_points_nominal_median']]